# 4. Train CatBoost (optional fast-inference layer)

Trains a hierarchical CatBoost classifier on whatever's in
`cat_predictions` for the active training schema (default: `three_level`,
source `ai_classify`). Writes a per-row prediction back into
`cat_predictions` with `source='catboost'` and a real `predict_proba`
confidence.

Why this lives in its own notebook:
* CatBoost is **optional** — the LLM strategies in notebook 3 already
  produce labels for every schema. CatBoost is a cheap fast-inference
  layer trained on those LLM labels.
* It's specific to schemas that have a clean L1/L2 hierarchy AND enough
  bootstrap labels per class. Today that's `three_level`.
* Skipping this notebook does not break the rest of the pipeline.

Tested on Serverless v4.

In [ ]:
%pip install pyyaml openpyxl pydantic catboost scikit-learn
%restart_python

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

from src.utils import get_spark
from src.config import load_config
import pyspark.sql.functions as F

spark = get_spark()
config = load_config()

In [ ]:
dbutils.widgets.removeAll()
dbutils.widgets.text('catalog', config.catalog)
dbutils.widgets.text('schema', config.schema_name)
dbutils.widgets.text('invoices_table', config.invoices)
dbutils.widgets.text('train_schema', 'three_level')
dbutils.widgets.text('train_source', 'ai_classify')
dbutils.widgets.text('run_catboost', 'true')

catalog = dbutils.widgets.get('catalog')
schema = dbutils.widgets.get('schema')
invoices_table = dbutils.widgets.get('invoices_table')
train_schema = dbutils.widgets.get('train_schema')
train_source = dbutils.widgets.get('train_source')
run_catboost = dbutils.widgets.get('run_catboost').lower() == 'true'
spark.sql(f'USE {catalog}.{schema}')

## Train two CatBoosts (L1, L2) and predict on a held-out test split

* L1 model: predicts level_path[0]
* L2 model: predicts level_path[1]
* Confidence = `m_l2.predict_proba(...).max(axis=1)` — already 0–1.

Skipped silently if `run_catboost=false` or if there isn't enough data
for the chosen `(train_schema, train_source)`.

In [ ]:
if run_catboost:
    import pandas as pd
    from catboost import CatBoostClassifier
    from sklearn.model_selection import train_test_split

    df = spark.sql(f'''
      SELECT i.order_id,
             CONCAT_WS(' | ', i.supplier, i.description, i.cost_centre) AS text,
             p.level_path[0] AS l1, p.level_path[1] AS l2
      FROM {catalog}.{schema}.{invoices_table} i
      JOIN {catalog}.{schema}.cat_predictions p
        ON p.order_id = i.order_id
       AND p.schema_name = '{train_schema}'
       AND p.source = '{train_source}'
      WHERE p.level_path[0] IS NOT NULL AND p.level_path[1] IS NOT NULL
    ''').toPandas()
    print(f'CatBoost training on {len(df):,} bootstrapped rows for schema={train_schema} source={train_source}')

    if len(df) < 50:
        print('Not enough labeled rows; skipping CatBoost.')
    else:
        train, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df['l1'])

        m_l1 = CatBoostClassifier(iterations=200, verbose=0, text_features=['text'])
        m_l1.fit(train[['text']], train['l1'])
        m_l2 = CatBoostClassifier(iterations=200, verbose=0, text_features=['text'])
        m_l2.fit(train[['text']], train['l2'])

        test_pred = test.copy()
        test_pred['pred_l1'] = m_l1.predict(test[['text']]).flatten()
        test_pred['pred_l2'] = m_l2.predict(test[['text']]).flatten()
        test_pred['confidence'] = m_l2.predict_proba(test[['text']]).max(axis=1)

        out = pd.DataFrame({
            'order_id': test_pred['order_id'].astype(str),
            'schema_name': train_schema,
            'code': test_pred['pred_l2'],
            'label': test_pred['pred_l2'],
            'level_path': [[a, b] for a, b in zip(test_pred['pred_l1'], test_pred['pred_l2'])],
            'confidence': test_pred['confidence'].astype(float),
            'rationale': 'CatBoost classifier predict_proba',
            'source': 'catboost',
            'candidates': [[] for _ in range(len(test_pred))],
        })
        sdf = spark.createDataFrame(out).withColumn('classified_at', F.current_timestamp())
        sdf.createOrReplaceTempView('_catboost_predictions')
        spark.sql(f'''
        MERGE INTO {catalog}.{schema}.cat_predictions t
        USING _catboost_predictions s
          ON t.order_id = s.order_id
         AND t.schema_name = s.schema_name
         AND t.source = s.source
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
        ''')
        print(f'Merged {len(out):,} catboost predictions')
else:
    print('Skipping CatBoost (run_catboost=false)')